In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# **Smart MCQ Solver — Baseline Model**
## TF-IDF + Cosine Similarity A simple, **untrained** reference point for the Smart MCQ Solver Challenge. For each question, the prompt and each of the 5 options are converted into TF-IDF vectors, then ranked by cosine similarity to the prompt — no modeltraining is involved.**Why this baseline matters:** it establishes a lower bound to judge whether our trained models (Logistic Regression, From-Scratch BiLSTM+Attention, ELECTRA) are actually learning meaningful patterns, or just matching keywords.


## Imports & Configuration
 

In [2]:
import os
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [3]:
DATA_PATH   = "/kaggle/input/competitions/smart-mcq-solver-challenge"
OPTION_COLS = ["A", "B", "C", "D", "E"]


## Load Dataset 

In [4]:
train = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
test  = pd.read_csv(os.path.join(DATA_PATH, "test.csv"))

print("Train shape:", train.shape)
print("Test shape :", test.shape)
train.head(5)


Train shape: (2000, 8)
Test shape : (500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


## **TF-IDF Vectorizer on All Text**
##  The vectorizer is fit on both prompts and all options together, so that option-specific words are included in the vocabulary too.

In [5]:
vectorizer_base = TfidfVectorizer(stop_words='english')

all_texts = []
for _, row in train.iterrows():
    all_texts.append(str(row['prompt']))
    for opt in OPTION_COLS:
        all_texts.append(str(row[opt]))

vectorizer_base.fit(all_texts)
print("Baseline TF-IDF vocabulary size:", len(vectorizer_base.vocabulary_))


Baseline TF-IDF vocabulary size: 2762


## **Define MAP@3 Metric and Similarity Ranking Function**

In [6]:
def map_at_3_baseline(ground_truth, predictions):
    score = 0.0
    for k, pred in enumerate(predictions[:3], start=1):
        if pred == ground_truth:
            score = 1.0 / k
            break
    return score


def rank_options_by_similarity(row, vectorizer):
    prompt_vec = vectorizer.transform([str(row['prompt'])])
    sims = {}
    for opt in OPTION_COLS:
        opt_vec = vectorizer.transform([str(row[opt])])
        sims[opt] = cosine_similarity(prompt_vec, opt_vec)[0][0]
    ranked = sorted(sims, key=sims.get, reverse=True)
    return ranked


## **Evaluate Baseline Performance (Train MAP@3)** 

In [7]:
baseline_scores = []
for _, row in train.iterrows():
    ranked = rank_options_by_similarity(row, vectorizer_base)
    baseline_scores.append(map_at_3_baseline(row['answer'], ranked))

baseline_map3 = np.mean(baseline_scores)
print(f"Baseline (TF-IDF + Cosine Similarity) Train MAP@3: {baseline_map3:.4f}")


Baseline (TF-IDF + Cosine Similarity) Train MAP@3: 0.3119


## Generate Submission File 

In [8]:
baseline_test_preds = []
for _, row in test.iterrows():
    ranked = rank_options_by_similarity(row, vectorizer_base)
    baseline_test_preds.append(' '.join(ranked[:3]))

baseline_submission = pd.DataFrame({
    'ID': test['id'],
    'Prediction': baseline_test_preds
})
baseline_submission.to_csv('submission.csv', index=False)
print("Baseline submission saved as submission.csv")
baseline_submission.head()


Baseline submission saved as submission.csv


,ID,Prediction
0,1,A B C
1,2,A B C
2,3,A D C
3,4,A E C
4,5,A C E


---
## **Summary**


| Metric | Value |
|---|---|
| Model | TF-IDF + Cosine Similarity |
| Training required | No |
| Train MAP@3 | ~0.31 |

## This score is intentionally low — it confirms that simple word-overlap is not sufficient to solve this dataset, motivating the need for trained models (Logistic Regression, From-Scratch BiLSTM+Attention, ELECTRA) that can capture deeper semantic understanding.
